# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 3: Neural Networks from Scratch
**JAWNVION LLC — AI Training Workbook**

TinyLlama has 1.1 billion parameters, but the math inside it is the same
as the tiny network you'll build in this chapter. Understanding this layer
means you will never be confused by a loss curve, a gradient explosion,
or a vanishing gradient again.

**What you'll build:**
- A single neuron (perceptron) — the atom of every neural network
- A two-layer MLP (Multi-Layer Perceptron) in pure NumPy
- Forward pass: input → hidden layer → output
- Backward pass: compute gradients with the chain rule
- Train the MLP on a binary classification task
- Visualise the decision boundary evolving during training

**No GPU required.**

In [ ]:
# — Cell 1: The Perceptron — One Neuron ——————————————
import numpy as np

# A single neuron does three things:
# 1. Weighted sum:    z = w1*x1 + w2*x2 + ... + wn*xn + bias
# 2. Activation:     a = sigmoid(z)   ← squashes to [0, 1]
# 3. Output:         a is the neuron's "firing" value

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Example: spam classifier neuron with 3 input features
features = {
    "contains_FREE":    1.0,   # yes
    "num_exclamation":  3.0,   # 3 exclamation marks
    "sender_known":     0.0,   # unknown sender
}
x = np.array(list(features.values()))

# Learned weights (what training would produce)
w = np.array([0.9, 0.4, -1.2])
b = -0.3

z = np.dot(w, x) + b          # weighted sum
a = sigmoid(z)                  # activation

print("SINGLE NEURON — Spam Classifier")
print("=" * 45)
for feat, val, wi in zip(features.keys(), x, w):
    print(f"  {feat:<22}  x={val:>4.1f}  w={wi:>5.2f}  contrib={val*wi:>6.3f}")
print(f"  {'bias':<22}  b={b:>4.1f}")
print(f"  {'─'*43}")
print(f"  Weighted sum  z = {z:.4f}")
print(f"  Sigmoid(z)    a = {a:.4f}")
print()
print(f"  Prediction: {'SPAM ✗' if a > 0.5 else 'NOT SPAM ✓'}  (threshold=0.5)")
print()
print("A neural network is just MANY of these neurons arranged in layers,")
print("each layer transforming the data in a learned way.")

In [ ]:
# — Cell 2: Multi-Layer Perceptron (MLP) Architecture ——
import numpy as np

# We'll build a 2-layer MLP:
#   Input (2 features) → Hidden (4 neurons) → Output (1 neuron)
#   This can learn ANY binary classification boundary.

np.random.seed(42)

class MLP:
    """Two-layer MLP with sigmoid activations, trained with gradient descent."""

    def __init__(self, n_input=2, n_hidden=4, n_output=1, lr=0.1):
        # Xavier initialisation — keeps gradients from exploding/vanishing
        scale1 = np.sqrt(2.0 / n_input)
        scale2 = np.sqrt(2.0 / n_hidden)
        self.W1 = np.random.randn(n_input, n_hidden)  * scale1
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_output) * scale2
        self.b2 = np.zeros((1, n_output))
        self.lr = lr

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        """Forward pass — compute predictions."""
        self.X   = X
        self.z1  = X @ self.W1 + self.b1          # (N, hidden)
        self.a1  = self.sigmoid(self.z1)            # (N, hidden)
        self.z2  = self.a1 @ self.W2 + self.b2     # (N, 1)
        self.a2  = self.sigmoid(self.z2)            # (N, 1) — predictions
        return self.a2

    def loss(self, y_true):
        """Binary cross-entropy — the correct loss for binary classification."""
        eps = 1e-8
        a   = np.clip(self.a2, eps, 1-eps)
        return -np.mean(y_true * np.log(a) + (1-y_true) * np.log(1-a))

    def backward(self, y_true):
        """Backward pass — compute gradients using the chain rule."""
        N  = y_true.shape[0]
        # Output layer gradients
        dz2 = self.a2 - y_true                       # (N, 1)
        dW2 = (self.a1.T @ dz2) / N
        db2 = dz2.mean(axis=0, keepdims=True)
        # Hidden layer gradients (chain rule through sigmoid)
        da1 = dz2 @ self.W2.T                        # (N, hidden)
        dz1 = da1 * self.a1 * (1 - self.a1)          # sigmoid derivative
        dW1 = (self.X.T @ dz1) / N
        db1 = dz1.mean(axis=0, keepdims=True)
        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def accuracy(self, X, y):
        preds = (self.forward(X) > 0.5).astype(float)
        return (preds == y).mean()

print("MLP architecture:")
model = MLP(n_input=2, n_hidden=4, n_output=1, lr=0.5)
print(f"  Layer 1 — W1: {model.W1.shape}  b1: {model.b1.shape}")
print(f"  Layer 2 — W2: {model.W2.shape}  b2: {model.b2.shape}")
params = model.W1.size + model.b1.size + model.W2.size + model.b2.size
print(f"  Total parameters: {params}")
print()
print("TinyLlama has 1,100,048,384 parameters — same math, vastly more layers.")

In [ ]:
# — Cell 3: Dataset — The Circle Classification Problem ——
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Generate two interleaved rings — NOT linearly separable.
# A single neuron cannot solve this. A 2-layer MLP can.
N = 300
angle = np.linspace(0, 2*np.pi, N//2)

# Inner ring (class 0)
r0 = np.random.uniform(0.0, 0.4, N//2)
X0 = np.column_stack([r0 * np.cos(angle), r0 * np.sin(angle)])
y0 = np.zeros((N//2, 1))

# Outer ring (class 1)
r1 = np.random.uniform(0.6, 1.0, N//2)
X1 = np.column_stack([r1 * np.cos(angle), r1 * np.sin(angle)])
y1 = np.ones((N//2, 1))

X = np.vstack([X0, X1])
y = np.vstack([y0, y1])

# Shuffle
idx = np.random.permutation(N)
X, y = X[idx], y[idx]

# Train / val split (80/20)
split = int(0.8 * N)
X_train, y_train = X[:split], y[:split]
X_val,   y_val   = X[split:], y[split:]

# Visualise
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X[y.ravel()==0, 0], X[y.ravel()==0, 1],
           color='#3498db', alpha=0.6, s=25, label='Class 0 (inner)')
ax.scatter(X[y.ravel()==1, 0], X[y.ravel()==1, 1],
           color='#e74c3c', alpha=0.6, s=25, label='Class 1 (outer)')
ax.set_title('Binary Classification Dataset\n(not linearly separable — needs hidden layer)',
             fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('/content/ch3_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"✓  Dataset: {N} samples, {split} train, {N-split} val")
print("   A single line CANNOT separate these — but a 2-layer MLP can.")

In [ ]:
# — Cell 4: Training the MLP ————————————————————————
import numpy as np

np.random.seed(42)

model     = MLP(n_input=2, n_hidden=8, n_output=1, lr=1.0)
EPOCHS    = 2000
BATCH     = 64
N_TRAIN   = X_train.shape[0]

train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

for epoch in range(EPOCHS):
    # Mini-batch gradient descent
    epoch_loss = 0.0
    idx        = np.random.permutation(N_TRAIN)
    for start in range(0, N_TRAIN, BATCH):
        batch = idx[start:start+BATCH]
        model.forward(X_train[batch])
        epoch_loss += model.loss(y_train[batch]) * len(batch)
        model.backward(y_train[batch])

    train_losses.append(epoch_loss / N_TRAIN)
    val_losses.append(model.loss(model.forward(X_val) or X_val) if False else
                      (_ := model.forward(X_val), model.loss(y_val))[1])
    train_accs.append(model.accuracy(X_train, y_train))
    val_accs.append(model.accuracy(X_val, y_val))

    if epoch % 400 == 0 or epoch == EPOCHS - 1:
        print(f"  Epoch {epoch:>4d} | train_loss={train_losses[-1]:.4f} "
              f"val_loss={val_losses[-1]:.4f} | "
              f"train_acc={train_accs[-1]:.3f} val_acc={val_accs[-1]:.3f}")

print()
print(f"✓  Training complete!")
print(f"   Final train accuracy : {train_accs[-1]*100:.1f}%")
print(f"   Final val accuracy   : {val_accs[-1]*100:.1f}%")

In [ ]:
# — Cell 5: Training Curves & Decision Boundary ————————
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Loss curves
axes[0].plot(train_losses, color='#3498db', linewidth=1.5, label='Train loss')
axes[0].plot(val_losses,   color='#e74c3c', linewidth=1.5, label='Val loss',  linestyle='--')
axes[0].set_title('Loss Curves', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Accuracy curves
axes[1].plot(train_accs, color='#3498db', linewidth=1.5, label='Train acc')
axes[1].plot(val_accs,   color='#e74c3c', linewidth=1.5, label='Val acc', linestyle='--')
axes[1].axhline(1.0, color='gray', linestyle=':', linewidth=1)
axes[1].set_title('Accuracy Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.4, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

# 3. Decision boundary
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 200),
                     np.linspace(-1.2, 1.2, 200))
grid   = np.column_stack([xx.ravel(), yy.ravel()])
probs  = model.forward(grid).reshape(xx.shape)

axes[2].contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.7)
axes[2].contour( xx, yy, probs, levels=[0.5], colors='white', linewidths=2)
axes[2].scatter(X_val[y_val.ravel()==0, 0], X_val[y_val.ravel()==0, 1],
                color='#3498db', s=30, edgecolors='white', linewidths=0.5,
                label='Class 0 (val)', zorder=3)
axes[2].scatter(X_val[y_val.ravel()==1, 0], X_val[y_val.ravel()==1, 1],
                color='#e74c3c', s=30, edgecolors='white', linewidths=0.5,
                label='Class 1 (val)', zorder=3)
axes[2].set_title(f'Learned Decision Boundary\n(val acc={val_accs[-1]*100:.1f}%)',
                   fontsize=12, fontweight='bold')
axes[2].legend(fontsize=8)
axes[2].set_aspect('equal')

plt.tight_layout()
plt.savefig('/content/ch3_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Results saved to /content/ch3_results.png")

In [ ]:
# — Cell 6: Activation Functions — Why Non-Linearity Matters
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 300)

activations = {
    "Sigmoid": (lambda x: 1/(1+np.exp(-np.clip(x,-500,500))), '#3498db',
                "Squashes to (0,1). Used in output for probability. Vanishing gradient risk."),
    "Tanh":    (lambda x: np.tanh(x), '#e74c3c',
                "Squashes to (-1,1). Zero-centred — better than sigmoid for hidden layers."),
    "ReLU":    (lambda x: np.maximum(0, x), '#2ecc71',
                "Most common hidden activation. Fast, no vanishing gradient (mostly)."),
    "GELU":    (lambda x: x * 0.5*(1 + np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3))),
                '#9b59b6',
                "Used in GPT/TinyLlama. Smooth version of ReLU — better for transformers."),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Activation Functions — Choosing Non-Linearity", fontsize=13, fontweight='bold')

for ax, (name, (fn, color, desc)) in zip(axes, activations.items()):
    ax.plot(z, fn(z), color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('z  (pre-activation)')
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.3)
    # wrap description
    words = desc.split()
    line, lines = [], []
    for w in words:
        line.append(w)
        if len(' '.join(line)) > 32:
            lines.append(' '.join(line[:-1]))
            line = [w]
    lines.append(' '.join(line))
    ax.set_xlabel('\n'.join(lines), fontsize=7.5)

plt.tight_layout()
plt.savefig('/content/ch3_activations.png', dpi=120, bbox_inches='tight')
plt.show()
print("✓  Plot saved to /content/ch3_activations.png")
print()
print("TinyLlama uses GELU activations in its feed-forward layers.")
print("When you look at its config.json in Ch5, you'll see 'hidden_act': 'silu'")
print("— SiLU (Swish) is a close relative of GELU.")

In [ ]:
# — Cell 7: How This Connects to LLMs ———————————————
print("FROM 3-LAYER MLP → TINYLLAMA — SAME MATH, DIFFERENT SCALE")
print("=" * 60)
print()
comparison = [
    ("Concept",              "Our MLP (Ch3)",                "TinyLlama 1.1B"),
    ("Parameters",           "25",                            "1,100,048,384"),
    ("Layers",               "2 (hidden + output)",           "22 transformer blocks"),
    ("Activation fn",        "Sigmoid",                       "SiLU (≈GELU)"),
    ("Weight matrices",      "W1 (2×8), W2 (8×1)",           "22× Q,K,V,O,gate,up,down"),
    ("Loss function",        "Binary cross-entropy",          "Cross-entropy (next token)"),
    ("Optimiser",            "Vanilla gradient descent",      "AdamW"),
    ("Learning rate",        "1.0 (aggressive, tiny model)",  "2e-4 (QLoRA fine-tune, Ch6)"),
    ("Batch size",           "64 samples",                    "4 sequences × 2048 tokens"),
    ("Training data",        "300 2D points",                 "Billions of text tokens"),
    ("Forward pass",         "z=Wx+b → sigmoid",             "Attention + FFN × 22 layers"),
    ("Backward pass",        "Chain rule (manual)",           "PyTorch autograd (automatic)"),
]
col_w = [26, 28, 22]
header = comparison[0]
print(f"  {header[0]:<{col_w[0]}} {header[1]:<{col_w[1]}} {header[2]:<{col_w[2]}}")
print(f"  {'─'*col_w[0]} {'─'*col_w[1]} {'─'*col_w[2]}")
for row in comparison[1:]:
    print(f"  {row[0]:<{col_w[0]}} {row[1]:<{col_w[1]}} {row[2]:<{col_w[2]}}")
print()
print("Every paper you read, every framework you use, every bug you debug")
print("traces back to the forward pass + backward pass you built in this chapter.")

## ✓ Chapter 3 Complete

You built a neural network from scratch, trained it with gradient descent,
and watched it learn a non-linear decision boundary — all without any ML framework.

**Next:** Chapter 4 — Transformers & Attention
You'll learn the architecture that powers every modern LLM: scaled dot-product
attention, multi-head attention, and the full Transformer block.

---
*JAWNVION LLC AI Training Workbook · peter@jawnvion.com*